In [ ]:
import os
import subprocess
import sys
import tempfile
from functools import partial

sys.path.insert(0, "/workspace/src")

import fsspec
import matplotlib.pyplot as plt
import numpy as np
import torch

from datasets.cwb import get_zarr_dataset  # noqa: F401
from inference.visualization import (
    animate_variable,
    animate_wind_speed,
    plot_comparison,
    plot_ensemble,
    plot_variable,
)

from physicsnemo import Module
from physicsnemo.diffusion.generate import diffusion_step, regression_step
from physicsnemo.diffusion.samplers import deterministic_sampler

plt.rcParams["animation.embed_limit"] = 100  # MB

# Data: same staging logic as check_cwb_regression.ipynb.
GCS_DATA = "gs://norcorrdiff-us/taiwan_dataset/cwa_dataset/cwa_dataset_cut.zarr"
DATA_PATH = "/workspace/data/cwa_dataset_cut.zarr"  # persistent mount (~/ml-ds_data)

# Diffusion results and the regression checkpoint it was conditioned on.
DIFF_CKPT_DIR = "gs://norcorrdiff-us/results/diffusion_cwb_20260626_095313/checkpoints_diffusion"
REG_CKPT_DIR  = "gs://norcorrdiff-us/results/regression_cwb_20260623_130422/checkpoints_regression"

# Must match the training config (config_training_taiwan_diffusion_gcp.yaml).
HR_MEAN_CONDITIONING = True
N_MEMBERS = 4  # ensemble size for quick checks; increase for production runs

if not os.path.exists(DATA_PATH):
    print(f"Staging {GCS_DATA}\n     -> {DATA_PATH} (one-time, ~30 GB)...")
    subprocess.run(
        [sys.executable, "/workspace/scripts/stage_zarr.py",
         "--src", GCS_DATA, "--dst", DATA_PATH],
        check=True,
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

DATASET_KWARGS = dict(
    data_path=DATA_PATH,
    in_channels=[0, 1, 2, 3, 4, 9, 10, 11, 12, 17, 18, 19],
    out_channels=[0, 1, 2, 3],
    img_shape_x=128,
    img_shape_y=128,
    add_grid=False,
    ds_factor=1,
    train=False,
    all_times=False,   # 2021 validation split
)

dataset = get_zarr_dataset(**DATASET_KWARGS)

assert len(dataset) > 0, (
    "Validation split is empty — the cut store may not span year 2021."
)

print(f"Dataset length : {len(dataset)}")
print(f"Image shape    : {dataset.image_shape()}")
print(f"Time range     : {dataset.time()[0]}  ->  {dataset.time()[-1]}")

In [ ]:
# Input (ERA5) and output (CWB) channels
input_channels  = dataset.input_channels()
output_channels = dataset.output_channels()

print(f"Input channels  ({len(input_channels)}):")
for ch in input_channels:
    print(f"  {ch.name:<40} level={ch.level}")

print(f"\nOutput channels ({len(output_channels)}):")
for ch in output_channels:
    print(f"  {ch.name:<40} level={ch.level}")

In [ ]:
# Load regression and diffusion checkpoints from GCS.
# Checkpoints are named <ModelClass>.<...>.<nimg>.mdlus; pick the highest nimg.
fs = fsspec.filesystem("gs")

def _load_latest_ckpt(ckpt_dir: str) -> Module:
    """Download the highest-nimg .mdlus checkpoint to a temp file and load it."""
    ckpts = [p for p in fs.ls(ckpt_dir) if p.endswith(".mdlus")]
    assert ckpts, f"No .mdlus checkpoints found in {ckpt_dir}"
    latest = max(ckpts, key=lambda p: int(p.split(".")[-2]))
    print(f"Loading: gs://{latest}")
    with tempfile.NamedTemporaryFile(suffix=".mdlus", delete=False) as tmp:
        tmp_path = tmp.name
    try:
        with fs.open(latest, "rb") as remote_f, open(tmp_path, "wb") as local_f:
            local_f.write(remote_f.read())
        return Module.from_checkpoint(tmp_path)
    finally:
        os.unlink(tmp_path)


def _prep_net(net: Module) -> Module:
    net = net.eval().to(device).to(memory_format=torch.channels_last)
    if hasattr(net, "amp_mode"):
        net.amp_mode = False
    return net


net_reg = _prep_net(_load_latest_ckpt(REG_CKPT_DIR))
net_res = _prep_net(_load_latest_ckpt(DIFF_CKPT_DIR))
print("Both networks loaded")

# Sampler: deterministic ODE solver (fast, reproducible).
sampler_fn = partial(deterministic_sampler, num_steps=9, solver="euler", patching=None)

In [ ]:
C_out = len(dataset.output_channels())
H, W = dataset.image_shape()


@torch.no_grad()
def predict_diffusion(idx, n_members=N_MEMBERS):
    """Return (truth, ensemble), truth is (C, H, W), ensemble is (n_members, C, H, W) — physical units."""
    target, input_ = dataset[idx]
    img_lr = input_[None].to(device).float().to(memory_format=torch.channels_last)

    # Regression mean: used both as conditioning and as the base for ensemble members.
    image_reg = regression_step(
        net=net_reg,
        img_lr=img_lr,
        latents_shape=(n_members, C_out, H, W),
        lead_time_label=None,
    )  # (n_members, C_out, H, W)

    mean_hr = image_reg[0:1] if HR_MEAN_CONDITIONING else None

    image_res = diffusion_step(
        net=net_res,
        sampler_fn=sampler_fn,
        img_shape=(H, W),
        img_out_channels=C_out,
        rank_batches=[torch.arange(n_members)],
        img_lr=img_lr.expand(n_members, -1, -1, -1).to(memory_format=torch.channels_last),
        rank=0,
        device=device,
        mean_hr=mean_hr,
        lead_time_label=None,
    )  # (n_members, C_out, H, W)

    ensemble = image_reg + image_res  # regression mean + stochastic residuals

    truth = dataset.denormalize_output(target[None].numpy())[0]
    ensemble_phys = dataset.denormalize_output(ensemble.cpu().numpy())
    return truth, ensemble_phys


# Sanity check on the first validation sample.
_truth, _ens = predict_diffusion(0)
print(f"truth shape    : {_truth.shape}")
print(f"ensemble shape : {_ens.shape}  (n_members, C_out, H, W)")
print(f"finite values  : truth={np.isfinite(_truth).all()}, ensemble={np.isfinite(_ens).all()}")

In [ ]:
# Ensemble truth / mean / std / error for the first validation sample, temperature_2m.
truth, ensemble = predict_diffusion(0)
plot_ensemble(truth, ensemble, dataset.output_channels(), dataset.time()[0], channel_idx=1)

In [ ]:
# Regression mean vs truth for comparison (channel_idx=1 = temperature_2m).
plot_comparison(truth, ensemble.mean(axis=0), dataset.output_channels(), dataset.time()[0], channel_idx=1)

In [ ]:
# Aggregate per-channel metrics over the validation set using ensemble mean (physical units).
# Fewer samples than regression notebook since diffusion is slower.
N_EVAL = min(len(dataset), 50)
print(f"Evaluating {N_EVAL} of {len(dataset)} validation samples")

out_names = [f"{c.name} {c.level}".strip() for c in dataset.output_channels()]

sse = np.zeros(C_out)
sae = np.zeros(C_out)
sbe = np.zeros(C_out)
n_pix = 0

names_no_level = [c.name for c in dataset.output_channels()]
has_wind = "eastward_wind_10m" in names_no_level and "northward_wind_10m" in names_no_level
if has_wind:
    u_idx = names_no_level.index("eastward_wind_10m")
    v_idx = names_no_level.index("northward_wind_10m")
    ws_sse = 0.0

for i in range(N_EVAL):
    truth, ensemble = predict_diffusion(i)
    pred = ensemble.mean(axis=0)  # ensemble mean as point estimate
    err = pred - truth
    sse += (err ** 2).sum(axis=(1, 2))
    sae += np.abs(err).sum(axis=(1, 2))
    sbe += err.sum(axis=(1, 2))
    n_pix += truth.shape[1] * truth.shape[2]
    if has_wind:
        ws_t = np.sqrt(truth[u_idx] ** 2 + truth[v_idx] ** 2)
        ws_p = np.sqrt(pred[u_idx] ** 2 + pred[v_idx] ** 2)
        ws_sse += ((ws_p - ws_t) ** 2).sum()

rmse = np.sqrt(sse / n_pix)
mae  = sae / n_pix
bias = sbe / n_pix

print(f"\nValidation metrics over {N_EVAL} samples — ensemble mean (physical units):\n")
print(f"{'channel':<28} {'RMSE':>10} {'MAE':>10} {'bias':>10}")
print("-" * 60)
for name, r, m, b in zip(out_names, rmse, mae, bias):
    print(f"{name:<28} {r:10.4f} {m:10.4f} {b:10.4f}")

if has_wind:
    print(f"\n{'wind_speed_10m':<28} {np.sqrt(ws_sse / n_pix):10.4f}  (RMSE)")

In [ ]:
# Static plots: ERA5 input and CWB truth for a chosen sample.
sample_idx = 0
# in_channels=[0,1,2,3,4,9,10,11,12,17,18,19] — original channel 17 is at subset index 9
plot_variable(dataset, "era5", channel_idx=9, sample_idx=sample_idx)

In [ ]:
plot_variable(dataset, "cwb", channel_idx=1, sample_idx=sample_idx)

In [ ]:
# Animate ERA5 and CWB truth over a week-long window.
# Diffusion predictions are not animated here — per-frame ensemble inference is too slow.
animate_variable(dataset, "era5", channel_idx=9, start=24*7, n_steps=24*7)

In [ ]:
animate_variable(dataset, "cwb", channel_idx=1, start=24*7, n_steps=24*7)

In [ ]:
animate_wind_speed(dataset, "era5", start=24*7, n_steps=24*7)

In [ ]:
animate_wind_speed(dataset, "cwb", start=24*7, n_steps=24*7)